# Y1 · Arabic data (Youssef) · just press Runtime > Run all
CPU runtime is enough. No Google Drive needed: at the end a zip downloads to your laptop.
Then drag it into the shared TwinCheck folder on drive.google.com (data/processed/).

In [ ]:
import os
os.makedirs('/content/twincheck/src/data', exist_ok=True)
os.makedirs('/content/twincheck/data/raw', exist_ok=True)
%cd /content/twincheck

In [ ]:
%%writefile /content/twincheck/src/data/download_m4fc_images.py
"""Download M4FC images for the vision side of TwinCheck (task Y1, hand-off to Feba F2).

What it does
    Reads M4FC.json (UKPLab/M4FC), picks the rows to download (Arabic claims, the
    official test split, or everything), downloads each image from `image_url` and
    falls back to `wayback_image_url`, checks it opens with PIL, and saves it as an
    RGB JPEG named after the row's stable TwinCheck id (m4fc_000123.jpg).
    Re-running skips images that are already saved, so it is safe after a disconnect.

Inputs
    --json     path to M4FC.json (from `git clone https://github.com/UKPLab/M4FC`)
    --subset   arabic | test | all   (default: arabic)
    --limit    only the first N selected rows (use 10 for the test run)

Outputs
    <out_dir>/m4fc_000123.jpg          one file per image
    <out_dir>/manifest.csv             id, image_path, claim_language, split, status, url_used

How to run
    python src/data/download_m4fc_images.py --json data/raw/M4FC/data/M4FC.json \
        --out_dir data/processed/vision/m4fc_images --subset arabic --limit 10
"""

import argparse
import csv
import io
import json
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
from PIL import Image
from tqdm import tqdm

MISSING = "not enough information"  # M4FC's value for empty fields
HEADERS = {"User-Agent": "Mozilla/5.0 (research dataset download; TwinCheck student project)"}


def m4fc_id(idx):
    """Return the stable TwinCheck id for row `idx` of M4FC.json, e.g. m4fc_000123."""
    return f"m4fc_{idx:06d}"


def select_rows(rows, subset):
    """Pick which M4FC rows to download.

    Args:
        rows: list of dicts loaded from M4FC.json.
        subset: "arabic" (claim_language == Arabic), "test" (official test split,
            Arabic rows first) or "all" (Arabic first, then test, then the rest).
    Returns:
        list of (idx, row) tuples in download order.
    """
    indexed = list(enumerate(rows))
    arabic = [(i, r) for i, r in indexed if r.get("claim_language") == "Arabic"]
    if subset == "arabic":
        return arabic
    arabic_ids = {i for i, _ in arabic}
    test = [(i, r) for i, r in indexed if r.get("split") == "test" and i not in arabic_ids]
    if subset == "test":
        return arabic + test
    seen = arabic_ids | {i for i, _ in test}
    return arabic + test + [(i, r) for i, r in indexed if i not in seen]


def fetch_image(url, timeout=20):
    """Download one URL and return a PIL RGB image, or None if it fails or is not an image."""
    if not url or url == MISSING:
        return None
    try:
        resp = requests.get(url, headers=HEADERS, timeout=timeout)
        if resp.status_code != 200 or len(resp.content) < 1000:
            return None
        img = Image.open(io.BytesIO(resp.content))
        img.load()  # forces a full decode, so broken files fail here
        return img.convert("RGB")
    except Exception:
        return None


def download_one(idx, row, out_dir):
    """Download one M4FC image (original URL, then Wayback copy) and save it as JPEG.

    Returns:
        dict for the manifest: id, image_path, claim_language, split, status, url_used.
        status is "ok", "exists" or "failed".
    """
    rid = m4fc_id(idx)
    path = os.path.join(out_dir, f"{rid}.jpg")
    record = {"id": rid, "image_path": path, "claim_language": row.get("claim_language"),
              "split": row.get("split"), "status": "failed", "url_used": ""}
    if os.path.exists(path) and os.path.getsize(path) > 0:
        record["status"] = "exists"
        return record
    for key in ("image_url", "wayback_image_url"):
        img = fetch_image(row.get(key))
        if img is not None:
            img.save(path, "JPEG", quality=95)
            record.update(status="ok", url_used=row.get(key))
            break
    return record


def main():
    """Parse arguments, download the selected images in parallel, write manifest.csv and print counts."""
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--json", required=True)
    ap.add_argument("--out_dir", default="data/processed/vision/m4fc_images")
    ap.add_argument("--subset", choices=["arabic", "test", "all"], default="arabic")
    ap.add_argument("--limit", type=int, default=None)
    ap.add_argument("--workers", type=int, default=8)
    args = ap.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)
    rows = json.load(open(args.json, encoding="utf-8"))
    todo = select_rows(rows, args.subset)[: args.limit]

    results = []
    with ThreadPoolExecutor(max_workers=args.workers) as pool:
        futures = [pool.submit(download_one, i, r, args.out_dir) for i, r in todo]
        for fut in tqdm(as_completed(futures), total=len(futures), desc="images"):
            results.append(fut.result())
    results.sort(key=lambda r: r["id"])

    # Merge with an existing manifest so runs with different subsets add up.
    manifest = os.path.join(args.out_dir, "manifest.csv")
    old = {}
    if os.path.exists(manifest):
        with open(manifest, encoding="utf-8") as f:
            old = {r["id"]: r for r in csv.DictReader(f)}
    for r in results:
        if r["status"] == "exists" and r["id"] in old:
            continue  # keep the original "ok" row and its URL
        old[r["id"]] = r
    with open(manifest, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["id", "image_path", "claim_language", "split", "status", "url_used"])
        w.writeheader()
        w.writerows(sorted(old.values(), key=lambda r: r["id"]))

    counts = {}
    for r in results:
        counts[r["status"]] = counts.get(r["status"], 0) + 1
    print(f"subset={args.subset} selected={len(todo)} " + " ".join(f"{k}={v}" for k, v in sorted(counts.items())))
    failed = [r["id"] for r in results if r["status"] == "failed"]
    if failed:
        print("failed ids (first 20):", failed[:20])


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/twincheck/src/data/load_xfact.py
"""Convert the Arabic part of X-FACT to the TwinCheck shared record format (task Y1).

What it does
    Reads X-FACT's train/dev/test TSV files (GitHub utahnlp/x-fact, folder
    data/x-fact/), keeps rows with language == "ar", maps X-FACT's coarse labels to
    supported / contradicted / unverified (dropping the rest), turns evidence_1..5
    and link_1..5 into the shared evidence list, and writes one JSONL file per split.
    X-FACT has real labels for its own splits, so we keep them (dev -> "val").
    ood.tsv and zeroshot.tsv contain no Arabic, so they are not read.

Inputs
    --raw_dir   folder with train.all.tsv, dev.all.tsv, test.all.tsv
    --limit     only the first N Arabic rows of each split (use 10 for the test run)

Outputs
    <out_dir>/xfact_ar_train.jsonl, xfact_ar_val.jsonl, xfact_ar_test.jsonl
    These are uncleaned; Rathi's src/data/clean.py writes the final
    data/processed/ar/ files.

How to run
    git clone --depth 1 https://github.com/utahnlp/x-fact data/raw/x-fact
    python src/data/load_xfact.py --raw_dir data/raw/x-fact/data/x-fact \
        --out_dir data/processed/ar/uncleaned --limit 10
"""

import argparse
import csv
import json
import os
from collections import Counter
from urllib.parse import urlparse

import pandas as pd

# X-FACT already maps every site's verdict to 7 coarse English labels.
# This follows the team label map in CLAUDE.md section 4 (same rows go in data/label_map.csv).
LABEL_MAP = {
    "true": "supported",
    "mostly true": "supported",
    "false": "contradicted",
    "mostly false": "contradicted",
    "complicated/hard to categorise": "unverified",  # includes "unproven", "unverifiable"
    "partly true/misleading": None,  # dropped (half true / misleading)
    "other": None,                   # dropped (satire, myth, clarification ...)
}
SPLITS = {"train": "train.all.tsv", "val": "dev.all.tsv", "test": "test.all.tsv"}
N_EVIDENCE = 5


def read_xfact_arabic(raw_dir, limit=None):
    """Read X-FACT TSVs and keep only Arabic rows.

    Args:
        raw_dir: folder that holds train.all.tsv, dev.all.tsv, test.all.tsv.
        limit: keep only the first `limit` Arabic rows per split (None = all).
    Returns:
        dict split -> DataFrame (all columns as strings, empty string for missing).
    """
    out = {}
    for split, fname in SPLITS.items():
        # QUOTE_NONE: snippets contain quote characters that would otherwise break parsing
        df = pd.read_csv(os.path.join(raw_dir, fname), sep="\t", quoting=csv.QUOTE_NONE,
                         dtype=str, keep_default_na=False)
        df = df[df["language"] == "ar"].reset_index(drop=True)
        out[split] = df.head(limit) if limit else df
    return out


def domain_of(url):
    """Return the domain of a URL without 'www.', or None for empty / NO_LINK values."""
    if not url or url == "NO_LINK":
        return None
    netloc = urlparse(url).netloc.lower()
    return netloc[4:] if netloc.startswith("www.") else netloc or None


def build_evidence(row):
    """Turn evidence_1..5 / link_1..5 of one X-FACT row into the shared evidence list."""
    evidence = []
    for i in range(1, N_EVIDENCE + 1):
        text = row.get(f"evidence_{i}", "").strip()
        if not text:
            continue
        url = row.get(f"link_{i}", "").strip()
        url = None if url in ("", "NO_LINK") else url
        evidence.append({"text": text, "url": url, "domain": domain_of(url)})
    return evidence


def row_to_record(row, number, split):
    """Convert one X-FACT row to a shared-format record.

    Args:
        row: dict for one TSV row.
        number: running number used in the id (xfact_ar_000001), fixed by file order.
        split: "train", "val" or "test".
    Returns:
        record dict, or None if the label is dropped by the label map.
    """
    orig = row["label"].strip().lower()
    label = LABEL_MAP.get(orig)
    if label is None:
        return None
    return {
        "id": f"xfact_ar_{number:06d}",
        "source": "xfact",
        "split": split,
        "language": "ar",
        "claim": row["claim"].strip(),
        "evidence": build_evidence(row),
        "image_path": None,
        "label": label,
        "orig_label": orig,
    }


def main():
    """Convert all splits, write JSONL files and print counts per split and label."""
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--raw_dir", required=True)
    ap.add_argument("--out_dir", default="data/processed/ar/uncleaned")
    ap.add_argument("--limit", type=int, default=None)
    args = ap.parse_args()
    os.makedirs(args.out_dir, exist_ok=True)

    number = 0  # numbers run across splits so every id is unique
    for split, df in read_xfact_arabic(args.raw_dir, args.limit).items():
        kept, dropped = [], Counter()
        for row in df.to_dict("records"):
            number += 1
            rec = row_to_record(row, number, split)
            if rec is None:
                dropped[row["label"]] += 1
            else:
                kept.append(rec)
        path = os.path.join(args.out_dir, f"xfact_ar_{split}.jsonl")
        with open(path, "w", encoding="utf-8") as f:
            for rec in kept:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        labels = Counter(r["label"] for r in kept)
        no_ev = sum(1 for r in kept if not r["evidence"])
        print(f"{split}: read={len(df)} kept={len(kept)} {dict(labels)} "
              f"dropped={sum(dropped.values())} {dict(dropped)} no_evidence={no_ev} -> {path}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/twincheck/src/data/load_m4fc.py
"""Convert the Arabic claims of M4FC to the TwinCheck shared record format (task Y1).

What it does
    Reads M4FC.json (GitHub UKPLab/M4FC, CC-BY-SA-4.0), keeps rows whose
    claim_language is Arabic, and writes them as TEST data only (agreed: nobody
    trains on M4FC, so all 614 Arabic rows are used for testing; the official M4FC
    split is not needed).
    - The Arabic claim (`multilingual_claim`) becomes `claim`; the fact-checker's
      verdict gives the label.
    - M4FC is almost all false claims. Following M4FC's own balanced setting, rows
      with use_true_caption == True also give a SUPPORTED claim: the Arabic true
      caption written for the same image (id m4fc_tc_000123).
    - M4FC has no evidence text, so the main file has evidence = [].
      A second file adds M4FC's context items (source, date, location, event ...)
      as one English snippet with domain "m4fc_context". This context comes from
      the fact-check article, so it is near-gold evidence and must be reported that way.

Inputs
    --json        path to M4FC.json
    --image_dir   folder used by download_m4fc_images.py (for image_path)
    --limit       only the first N Arabic rows (use 10 for the test run)

Outputs (in --out_dir)
    m4fc_ar_test.jsonl        main file, evidence = []
    m4fc_ar_test_ctx.jsonl    same rows, evidence = M4FC context snippet
    m4fc_ar_claims_en.csv     id, English claim (for cross-lingual checks in Y4/Y5)

How to run
    git clone --depth 1 https://github.com/UKPLab/M4FC data/raw/M4FC
    python src/data/load_m4fc.py --json data/raw/M4FC/data/M4FC.json \
        --out_dir data/processed/ar/uncleaned --limit 10
"""

import argparse
import csv
import json
import os
from collections import Counter

MISSING = "not enough information"  # M4FC's value for empty fields
LABEL_MAP = {  # M4FC fine verdict -> team label (same rows go in data/label_map.csv)
    "true": "supported",
    "false (out-of-context)": "contradicted",
    "false (manipulated/fake)": "contradicted",
    "false (ai-generated)": "contradicted",
    "not enough information": "unverified",
}
CONTEXT_FIELDS = ["source", "date", "location", "event", "motivation"]


def read_m4fc_arabic(json_path, limit=None):
    """Load M4FC.json and return (idx, row) pairs for Arabic claims.

    idx is the row's position in M4FC.json; it is used in the id so ids never change
    and match the image file names from download_m4fc_images.py.
    """
    rows = json.load(open(json_path, encoding="utf-8"))
    arabic = [(i, r) for i, r in enumerate(rows) if r.get("claim_language") == "Arabic"]
    return arabic[:limit] if limit else arabic


def image_path_for(idx, image_dir):
    """Return the image path Feba's folder uses for M4FC row idx."""
    return f"{image_dir.rstrip('/')}/m4fc_{idx:06d}.jpg"


def context_evidence(row):
    """Build one evidence snippet from M4FC's context items, or [] if all are missing."""
    parts = [f"{k.capitalize()}: {row[k]}" for k in CONTEXT_FIELDS
             if isinstance(row.get(k), str) and row[k].strip() and row[k] != MISSING]
    if not parts:
        return []
    return [{"text": ". ".join(parts) + ".", "url": row.get("article_url"), "domain": "m4fc_context"}]


def map_label(verdict):
    """Map an M4FC fine verdict to supported/contradicted/unverified (None if unknown)."""
    verdict = verdict.strip().lower()
    if verdict in LABEL_MAP:
        return LABEL_MAP[verdict]
    return "contradicted" if verdict.startswith("false") else None


def row_to_record(idx, row, image_dir):
    """Convert one Arabic M4FC row to a shared-format record (or None if the label is unknown)."""
    label = map_label(row["verdict"])
    if label is None:
        return None
    return {
        "id": f"m4fc_{idx:06d}",
        "source": "m4fc",
        "split": "test",
        "language": "ar",
        "claim": row["multilingual_claim"].strip(),
        "evidence": [],
        "image_path": image_path_for(idx, image_dir),
        "label": label,
        "orig_label": row["verdict"].strip().lower(),
    }


def true_caption_record(idx, row, image_dir):
    """Make a SUPPORTED record from the Arabic true caption (M4FC balanced setting), or None."""
    caption = row.get("multilingual_true_caption", MISSING)
    if not row.get("use_true_caption") or not caption or caption == MISSING:
        return None
    return {
        "id": f"m4fc_tc_{idx:06d}",
        "source": "m4fc",
        "split": "test",
        "language": "ar",
        "claim": caption.strip(),
        "evidence": [],
        "image_path": image_path_for(idx, image_dir),
        "label": "supported",
        "orig_label": "true_caption",
    }


def write_jsonl(path, records):
    """Write records as JSON Lines with Arabic kept readable (ensure_ascii=False)."""
    with open(path, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def main():
    """Convert Arabic M4FC rows, write the three output files and print counts."""
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--json", required=True)
    ap.add_argument("--image_dir", default="data/processed/vision/m4fc_images")
    ap.add_argument("--out_dir", default="data/processed/ar/uncleaned")
    ap.add_argument("--limit", type=int, default=None)
    args = ap.parse_args()
    os.makedirs(args.out_dir, exist_ok=True)

    main_recs, ctx_recs, en_rows, skipped = [], [], [], 0
    for idx, row in read_m4fc_arabic(args.json, args.limit):
        if map_label(row["verdict"]) is None:
            skipped += 1
        # (record, English version of the same claim) for the claim and its true caption
        pairs = [(row_to_record(idx, row, args.image_dir), row["claim"]),
                 (true_caption_record(idx, row, args.image_dir), row.get("true_caption", MISSING))]
        for rec, claim_en in pairs:
            if rec is None:
                continue
            main_recs.append(rec)
            ctx_recs.append({**rec, "evidence": context_evidence(row)})
            en_rows.append({"id": rec["id"], "claim_en": claim_en})

    write_jsonl(os.path.join(args.out_dir, "m4fc_ar_test.jsonl"), main_recs)
    write_jsonl(os.path.join(args.out_dir, "m4fc_ar_test_ctx.jsonl"), ctx_recs)
    with open(os.path.join(args.out_dir, "m4fc_ar_claims_en.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["id", "claim_en"])
        w.writeheader()
        w.writerows(en_rows)

    labels = Counter(r["label"] for r in main_recs)
    no_ctx = sum(1 for r in ctx_recs if not r["evidence"])
    print(f"m4fc arabic: records={len(main_recs)} {dict(labels)} skipped_unknown_label={skipped} "
          f"no_context_evidence={no_ctx} -> {args.out_dir}")


if __name__ == "__main__":
    main()

In [ ]:
!rm -rf data/raw/M4FC data/raw/x-fact
!git clone -q --depth 1 https://github.com/UKPLab/M4FC data/raw/M4FC
!git clone -q --depth 1 https://github.com/utahnlp/x-fact data/raw/x-fact
!ls data/raw/M4FC/data/M4FC.json data/raw/x-fact/data/x-fact/

## Images: 10-image test, then all 614 Arabic (5-10 min, keep the tab open)

In [ ]:
!python src/data/download_m4fc_images.py --json data/raw/M4FC/data/M4FC.json \
    --out_dir data/processed/vision/m4fc_images --subset arabic --limit 10
from IPython.display import Image as Show, display
import glob
for p in sorted(glob.glob('data/processed/vision/m4fc_images/*.jpg'))[:3]:
    display(Show(p, width=250))

In [ ]:
!python src/data/download_m4fc_images.py --json data/raw/M4FC/data/M4FC.json \
    --out_dir data/processed/vision/m4fc_images --subset arabic

## Arabic data files: 10-row test, then full

In [ ]:
!python src/data/load_xfact.py --raw_dir data/raw/x-fact/data/x-fact --out_dir /content/sample10 --limit 10
!python src/data/load_m4fc.py --json data/raw/M4FC/data/M4FC.json --out_dir /content/sample10 --limit 10
!python src/data/load_xfact.py --raw_dir data/raw/x-fact/data/x-fact --out_dir data/processed/ar/uncleaned
!python src/data/load_m4fc.py --json data/raw/M4FC/data/M4FC.json --out_dir data/processed/ar/uncleaned
!wc -l data/processed/ar/uncleaned/*

## Counts for the group message + download the zip

In [ ]:
import pandas as pd
m = pd.read_csv('data/processed/vision/m4fc_images/manifest.csv')
print('IMAGES:', m.groupby('status').size().to_dict())
!cd data/processed && zip -qr /content/Y1_outputs.zip vision/m4fc_images ar/uncleaned
!ls -lh /content/Y1_outputs.zip
from google.colab import files
files.download('/content/Y1_outputs.zip')